# 01 — Build Master Benchmark Dataset

This notebook constructs the unified benchmark dataset used throughout the study.

The original benchmark dataset is extended with DAv3c results while preserving the multi-run evaluation protocol used in the benchmark study. The notebook performs:

- Loading benchmark solver results
- Loading and aggregating DAv3c runs
- Runtime standardization
- Seed generation
- Dataset validation
- Export of the unified dataset

The resulting dataset serves as the primary input for all downstream analyses.

In [9]:
# Imports
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Project Imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# =========================================================
# Input Paths
# =========================================================

PATH_BENCHMARK = (
    cfg.RAW_DIR / "benchmark/all_runs.csv"
)

PATH_DAV3C = (
    cfg.RAW_DIR / "dav3c_runs"
)

# =========================================================
# Output Paths
# =========================================================

OUTPUT_PATH = (
    cfg.PROCESSED_DIR / "final_dataset.parquet"
)

In [11]:
# Load Benchmark Results

df_bench = pd.read_csv(PATH_BENCHMARK)

print(f"Benchmark rows: {len(df_bench):,}")

df_bench.head()

Benchmark rows: 98,415


,file,name,heuristic,seed,objective,runtime,solution
0,./BURER2002/G55-13.945-5.csv,G55,BURER2002,144,10243.0,13.964832,1110100000010001111111001011010010000001110001...
1,./BURER2002/G55-13.945-5.csv,G55,BURER2002,244,10247.0,13.946340,0000110011100100000100110000001001001111001100...
2,./BURER2002/G55-13.945-5.csv,G55,BURER2002,344,10249.0,13.960023,1010010111111000110000000101000001011100010100...
3,./BURER2002/G55-13.945-5.csv,G55,BURER2002,444,10243.0,13.963533,0001011001100011010000101111000101101000000110...
4,./BURER2002/G55-13.945-5.csv,G55,BURER2002,544,10233.0,13.970771,0011110101110000011100011000010001001111011011...


In [12]:
# Load DAv3c Result Files

dfs = []

for file in PATH_DAV3C.glob("*.csv"):

    df = pd.read_csv(
        file,
        dtype={"solution": str}
    )

    # Store source file for traceability
    df["file"] = f"./dav3c_runs/{file.name}"

    # Match benchmark heuristic naming
    df["heuristic"] = "DAv3c"

    dfs.append(df)

dav3c_df = pd.concat(
    dfs,
    ignore_index=True
)

print(f"DAv3c rows: {len(dav3c_df):,}")

dav3c_df.head()

DAv3c rows: 17,530


,name,experiment,duration,objective,limit,energy,qubo_value,int_only,scaling_action,scaling_precision,solution,file,heuristic
0,g003034,0,3028.663,20683.0,3.614,-20683.0,20683.0,True,NOTHING,64,1110101010001001111001111011111001100010110001...,./dav3c_runs/g003034-3.614-5.csv,DAv3c
1,g003034,1,3028.209,20683.0,3.614,-20683.0,20683.0,True,NOTHING,64,1110101010001001111001111011111001100010110001...,./dav3c_runs/g003034-3.614-5.csv,DAv3c
2,g003034,2,3032.698,20683.0,3.614,-20683.0,20683.0,True,NOTHING,64,1110101010001001111001111011111001100010110001...,./dav3c_runs/g003034-3.614-5.csv,DAv3c
3,g003034,3,3030.814,20683.0,3.614,-20683.0,20683.0,True,NOTHING,64,1110101010001001111001111011111001100010110001...,./dav3c_runs/g003034-3.614-5.csv,DAv3c
4,g003034,4,3028.171,20683.0,3.614,-20683.0,20683.0,True,NOTHING,64,1110101010001001111001111011111001100010110001...,./dav3c_runs/g003034-3.614-5.csv,DAv3c


In [13]:
# Runtime Standardization
dav3c_df["runtime"] = (
    dav3c_df["duration"] / 1000
)


# Seed Construction
dav3c_df["seed"] = (
    144 + 100 * dav3c_df["experiment"]
)

# Select Benchmark-Compatible Columns
dav3c_df = dav3c_df[
    [
        "file",
        "name",
        "heuristic",
        "seed",
        "objective",
        "runtime",
        "solution"
    ]
]

dav3c_df.head()

,file,name,heuristic,seed,objective,runtime,solution
0,./dav3c_runs/g003034-3.614-5.csv,g003034,DAv3c,144,20683.0,3.028663,1110101010001001111001111011111001100010110001...
1,./dav3c_runs/g003034-3.614-5.csv,g003034,DAv3c,244,20683.0,3.028209,1110101010001001111001111011111001100010110001...
2,./dav3c_runs/g003034-3.614-5.csv,g003034,DAv3c,344,20683.0,3.032698,1110101010001001111001111011111001100010110001...
3,./dav3c_runs/g003034-3.614-5.csv,g003034,DAv3c,444,20683.0,3.030814,1110101010001001111001111011111001100010110001...
4,./dav3c_runs/g003034-3.614-5.csv,g003034,DAv3c,544,20683.0,3.028171,1110101010001001111001111011111001100010110001...


In [14]:
# Merge Datasets
df_final = pd.concat(
    [df_bench, dav3c_df],
    ignore_index=True
)

print(f"Final dataset rows: {len(df_final):,}")

df_final.head()

Final dataset rows: 115,945


,file,name,heuristic,seed,objective,runtime,solution
0,./BURER2002/G55-13.945-5.csv,G55,BURER2002,144,10243.0,13.964832,1110100000010001111111001011010010000001110001...
1,./BURER2002/G55-13.945-5.csv,G55,BURER2002,244,10247.0,13.946340,0000110011100100000100110000001001001111001100...
2,./BURER2002/G55-13.945-5.csv,G55,BURER2002,344,10249.0,13.960023,1010010111111000110000000101000001011100010100...
3,./BURER2002/G55-13.945-5.csv,G55,BURER2002,444,10243.0,13.963533,0001011001100011010000101111000101101000000110...
4,./BURER2002/G55-13.945-5.csv,G55,BURER2002,544,10233.0,13.970771,0011110101110000011100011000010001001111011011...


In [15]:
# Validation

print("Unique heuristics:")
print(df_final["heuristic"].unique())

print("\nMissing values:")
print(df_final.isna().sum())

print("\nUnique instances:")
print(df_final["name"].nunique())

Unique heuristics:
<ArrowStringArray>
[          'BURER2002',                'DAv2',                'DAv3',
          'DWave-DAv3',           'GSet-DAv2',           'GSet-DAv3',
         'MERZ1999GLS', 'PALUBECKIS2004bMST2',               'DAv3c']
Length: 9, dtype: str

Missing values:
file         0
name         0
heuristic    0
seed         0
objective    0
runtime      0
solution     0
dtype: int64

Unique instances:
3515


In [20]:
# Export Dataset

df_final.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(f"Saved dataset to:\n{OUTPUT_PATH}")

Saved dataset to:
/Users/sargun/Desktop/DAMB-DAv3c/data/processed/final_dataset.parquet
